Theory

Few-shot prompting provides the model with k demonstration examples (input–output pairs) before the actual query. The model performs in-context learning (ICL) — inferring the task pattern from examples without any weight updates.

Research shows that example quality matters more than quantity. Diverse, representative examples covering edge cases outperform many similar examples. The model implicitly learns the output schema, reasoning style, and domain vocabulary from demonstrations.

In LangGraph, a dedicated example_selector node dynamically retrieves the most relevant examples from a vector store, enabling adaptive few-shot prompting at scale.

Best Practices

Use 3–8 examples (diminishing returns beyond that)

Ensure examples cover diverse edge cases

Order matters: hardest examples last works best

Examples must follow the exact target format

Use semantic search to pick relevant examples dynamically

Include a mix of positive and negative examples

🔹 What is SemanticSimilarityExampleSelector?

SemanticSimilarityExampleSelector is a dynamic few-shot example selector.

Instead of hardcoding examples in your prompt, it:

Stores a set of examples
Converts them into embeddings
At runtime, selects the most semantically similar examples to the user query

👉 In simple terms:

“Pick the most relevant examples based on meaning (not keywords).”

🔹 Why it matters (especially in LangGraph)

In LangGraph multi-agent systems, prompts are everything.
Agents behave better when given relevant context/examples.

Without selector ❌:

Static examples
Irrelevant context
Poor reasoning

With selector ✅:

Context adapts dynamically
Better accuracy
Smarter agents

In [12]:
# ─────────────────────────────────────────────────────────
# Few-Shot Prompting with Dynamic Example Selection
# Pattern: Vector store retrieval → Example injection → LLM
# ─────────────────────────────────────────────────────────

from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.vectorstores import FAISS
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
import json
import os
from dotenv import load_dotenv
from IPython.display import Image, display
load_dotenv()

True

In [13]:
# ── Example Bank (Production: load from database) ─────────
EXAMPLE_BANK = [
    {
        "input": "Extract entities from: 'Apple Inc. CEO Tim Cook announced iPhone 16 in Cupertino'",
        "output": json.dumps({"persons": ["Tim Cook"], "organizations": ["Apple Inc."],
                       "products": ["iPhone 16"], "locations": ["Cupertino"]})
    },
    {
        "input": "Extract entities from: 'NASA astronaut Sally Ride flew the Space Shuttle'",
        "output": json.dumps({"persons": ["Sally Ride"], "organizations": ["NASA"],
                       "products": ["Space Shuttle"], "locations": []})
    },
    {
        "input": "Extract entities from: 'Microsoft released Windows 11 in Redmond, Washington'",
        "output": json.dumps({"persons": [], "organizations": ["Microsoft"],
                       "products": ["Windows 11"], "locations": ["Redmond", "Washington"]})
    },
    {
        "input": "Extract entities from: 'Elon Musk founded SpaceX and Tesla Motors'",
        "output": json.dumps({"persons": ["Elon Musk"], "organizations": ["SpaceX", "Tesla Motors"],
                       "products": [], "locations": []})
    },
]

In [14]:
# ── State Schema ──────────────────────────────────────────
class FewShotState(TypedDict):
    query: str
    num_examples: int            # How many examples to select
    selected_examples: list      # Retrieved examples
    built_prompt: str
    response: str
    parsed_result: dict

In [15]:
# ── Semantic Example Selector ─────────────────────────────
def create_example_selector(examples: list, k: int = 3):
    """Creates a semantic similarity-based example selector."""
    embeddings = OpenAIEmbeddings()
    return SemanticSimilarityExampleSelector.from_examples(
        examples=examples,
        embeddings=embeddings,
        vectorstore_cls=FAISS,
        k=k,
        input_keys=["input"]
    )

In [16]:
# ── Node 1: Dynamic Example Selection ────────────────────
def select_examples(state: FewShotState) -> FewShotState:
    """
    Uses semantic similarity to pick the most relevant examples
    for the current query from the example bank.
    This is CRITICAL for production — random examples perform poorly.
    """
    selector = create_example_selector(EXAMPLE_BANK, k=state["num_examples"])
    selected = selector.select_examples({"input": state["query"]})
    return {**state, "selected_examples": selected}

In [ ]:
# ── Node 2: Build Few-Shot Prompt ─────────────────────────
def build_few_shot_prompt(state: FewShotState) -> FewShotState:
    """
    Assembles prompt with demonstrations using the 
    Input: / Output: delimiter pattern for maximum clarity.
    """
    examples_text = ""
    for i, ex in enumerate(state["selected_examples"], 1):
        examples_text += f"""
Example {i}:
Input: {ex['input']}
Output: {ex['output']}
---"""
# Example 1:
# Input: What is AI?
# Output: Artificial Intelligence is...
# ---

# Example 2:
# Input: What is ML?
# Output: Machine Learning is...
# ---

    prompt = f"""You are a Named Entity Recognition (NER) system.
Extract all entities and return valid JSON with keys:
persons, organizations, products, locations.

Here are {len(state['selected_examples'])} demonstration examples:
{examples_text}

Now extract entities from the following input.
Return ONLY the JSON object, no explanation.

Input: {state['query']}
Output:"""

    return {**state, "built_prompt": prompt}

In [18]:
# ── Node 3: Call LLM and Parse ───────────────────────────
def call_and_parse(state: FewShotState) -> FewShotState:
    llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
    response = llm.invoke([HumanMessage(content=state["built_prompt"])])
    raw = response.content.strip()
    parsed = json.loads(raw)
    return {**state, "response": raw, "parsed_result": parsed}

In [19]:
# ── Graph Construction ────────────────────────────────────
def build_few_shot_graph():
    builder = StateGraph(FewShotState)

    builder.add_node("select_examples", select_examples)
    builder.add_node("build_few_shot_prompt", build_few_shot_prompt)
    builder.add_node("call_and_parse", call_and_parse)

    builder.add_edge(START, "select_examples")
    builder.add_edge("select_examples", "build_few_shot_prompt")
    builder.add_edge("build_few_shot_prompt", "call_and_parse")
    builder.add_edge("call_and_parse", END)

    return builder.compile()

In [29]:
# ── Usage ─────────────────────────────────────────────────
graph = build_few_shot_graph()
# View
# display(Image(graph.get_graph().draw_mermaid_png()))

result = graph.invoke({
    "query": "Extract entities from: 'Google CEO Sundar Pichai announced Gemini AI in Mountain View'",
    "num_examples": 3,
    "selected_examples": [],
    "built_prompt": "",
    "response": "",
    "parsed_result": {}
})



In [31]:
print(result["built_prompt"])
print("*"*100)
print(result["parsed_result"])

You are a Named Entity Recognition (NER) system.
Extract all entities and return valid JSON with keys:
persons, organizations, products, locations.

Here are 3 demonstration examples:

Example 1:
Input: Extract entities from: 'Apple Inc. CEO Tim Cook announced iPhone 16 in Cupertino'
Output: {"persons": ["Tim Cook"], "organizations": ["Apple Inc."], "products": ["iPhone 16"], "locations": ["Cupertino"]}
---
Example 2:
Input: Extract entities from: 'Elon Musk founded SpaceX and Tesla Motors'
Output: {"persons": ["Elon Musk"], "organizations": ["SpaceX", "Tesla Motors"], "products": [], "locations": []}
---
Example 3:
Input: Extract entities from: 'Microsoft released Windows 11 in Redmond, Washington'
Output: {"persons": [], "organizations": ["Microsoft"], "products": ["Windows 11"], "locations": ["Redmond", "Washington"]}
---

Now extract entities from the following input.
Return ONLY the JSON object, no explanation.

Input: Extract entities from: 'Google CEO Sundar Pichai announced Gem